## Building a character tokenizer

Why a character tokenizer?

Our input is "10 + 5 * 3" and out put is "25". We have some known operators * / + -, and the digits 1234567890 positive and negative signs.
We can't use word token or semi-word token because a number 123 is different from 321 or 124 or 12 or 23. Semi-number don't make sense, and
mapping every number is not possible. Our embedding is discrete not continuos.

In [4]:
chars = "0123456789+-*/ .=;" # the ; is our <eos>
vocab = {char: i for i, char in enumerate(chars)}
print('Length of vocab: ', len(vocab), '\n', vocab)

Length of vocab:  18 
 {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '+': 10, '-': 11, '*': 12, '/': 13, ' ': 14, '.': 15, '=': 16, ';': 17}


In [17]:
import torch

embedding_dim = 20

embedding = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=embedding_dim)

print(embedding.weight[:2])
print(f"shape of embedding weights: {embedding.weight.shape}")
print(f"data type of embedding weights: {embedding.weight.dtype}")

tensor([[ 0.6182,  0.8708,  0.7305,  0.7321,  1.2045,  1.1602, -0.7541,  0.5428,
         -1.8693,  1.1661, -0.6062,  0.8970, -1.0936,  0.1379,  1.0652, -0.4521,
         -0.2280,  1.1307, -1.3107, -1.5159],
        [-1.5069,  0.6113,  0.5493, -0.9601, -1.6427, -1.0711,  0.2565,  1.3730,
         -1.2069,  0.6674,  0.3865,  0.7270, -0.7954,  0.2766, -1.2668,  0.2818,
         -0.6294, -0.6019, -2.4687, -0.1885]], grad_fn=<SliceBackward0>)
shape of embedding weights: torch.Size([18, 20])
data type of embedding weights: torch.float32


In [18]:
def input_to_token_ids(input_string: str) -> list[int]:
    return [vocab[char] for char in input_string]

In [19]:
def token_ids_to_embeddings(token_ids: list[int]) -> torch.Tensor:
    # convert the list of ints to a torch.LongTensor
    token_tensor = torch.tensor(token_ids).long()
    embedded_tokens = embedding(token_tensor)
    
    return embedded_tokens

In [20]:
# let's test it out
input_str = "100 + 5 * 3"

ids = input_to_token_ids(input_str)
print(f"numerical IDs for '{input_str}': {ids}")

embeddings = token_ids_to_embeddings(ids)
print(f"shape of embeddings: {embeddings.shape}")
print(f"first character '{input_str[0]}' embedding: {embeddings[0]}")

numerical IDs for '100 + 5 * 3': [1, 0, 0, 14, 10, 14, 5, 14, 12, 14, 3]
shape of embeddings: torch.Size([11, 20])
first character '1' embedding: tensor([-1.5069,  0.6113,  0.5493, -0.9601, -1.6427, -1.0711,  0.2565,  1.3730,
        -1.2069,  0.6674,  0.3865,  0.7270, -0.7954,  0.2766, -1.2668,  0.2818,
        -0.6294, -0.6019, -2.4687, -0.1885], grad_fn=<SelectBackward0>)


In [23]:
def input_to_embeddings(input_string: str) -> torch.Tensor:
    input_string += ' = '
    return token_ids_to_embeddings(input_to_token_ids(input_string))

def output_to_embeddings(output_string: str) -> torch.Tensor:
    output_string += ';' # this ; is <eos>
    return token_ids_to_embeddings(input_to_token_ids(output_string))

In [24]:
sample_input_string = "10 + 5 * 3"
sample_output_string = "25"

input_embedding = input_to_embeddings(sample_input_string)
output_embedding = output_to_embeddings(sample_output_string)

print(f"input string: '{sample_input_string}'")
print(f"input embedding shape: {input_embedding.shape}")
print(f"output string: '{sample_output_string}'")
print(f"output embedding shape: {output_embedding.shape}")

input string: '10 + 5 * 3'
input embedding shape: torch.Size([13, 20])
output string: '25'
output embedding shape: torch.Size([3, 20])
